# 1. The data, and rank first

*rang* is Dutch for **rank**, and the rank of a normal matrix is the first thing
this project measures. Before we calibrate, flag or image anything we ask a
plain question: **can these data determine the gains at all?**

This notebook

1. loads a small piece of real MeerKAT L-band data (four antennas, one scan) and
   checks that the loader reproduces an archived run exactly;
2. looks at amplitudes, flags and weights per baseline and channel;
3. runs the identifiability ("rank") survey on every gain solution bin and
   explains its verdicts and status codes.

Everything heavy runs in rang's Rust core; the functions in `rang.notebook`
only arrange inputs and draw. Every plot is a function you can also call from
a script.

In [ ]:
%matplotlib inline
import os

os.environ.setdefault("RAYON_NUM_THREADS", "4")  # share the cores; set before the first core call

import numpy as np
import pandas as pd

from rang.notebook import data, experiment, flags, gains, image, steer, survey

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

# Downloads and the extracted fixture go here; choose your own with RANG_NOTEBOOK_CACHE.
CACHE = data.default_cache_dir()
# LUNO sections run when a local copy of the slice exists (RANG_NOTEBOOK_LUNO=0 turns them off).
LUNO = os.environ.get("RANG_NOTEBOOK_LUNO", "1") != "0" and data.find_luno_slice() is not None
print("cache:", CACHE, "| LUNO slice available:", LUNO)

## The fixture

Scan 9 of the public SKA pointing fixture `outer_scans.zip` (SHA256 pinned): 140
rows = 14 integrations × 10 baselines (6 cross-correlations and 4
autocorrelations of m000–m003), 4096 channels from 856 to 1712 MHz, four
correlations XX, XY, YX, YY, on the calibrator J1939−6342 (PKS 1934−638). It
is a reference-pointing scan, not a science observation; we use it because it
is small and real.

The loader reads the Measurement Set with the core's own casacore-table reader
(no python-casacore) and builds the same *state* as the frozen preparation step
of the first experiment:

- a **model**: PKS 1934−638 as an unpolarized point source at the phase centre
  with the Reynolds (1994) spectrum, so XX = YY = I(ν) and XY = YX = 0;
- an **eligibility mask**: unflagged cross-correlation cells in channels
  1024–3071 with finite data and positive weights;
- a fixed **train / validation / test split** of those cells (about 80/10/10,
  seed 7). The solver only sees train; validation and test measure how well
  the gains *predict data they were not fitted to*.

If you do not have the archive, `data.download_fixture(CACHE)` fetches the
45 MB public file and checks its hash.

In [ ]:
ds = data.load_fixture(cache_dir=CACHE)
pd.Series(data.summary(ds), name="value").to_frame()

### What a professional checks first: is the loader faithful?

A new reader is only trusted once it reproduces something archived. The first
fixture campaign (2026-09, QuartiCal in a container) recorded its split
counts and its baseline solve. We rebuild the split and solve with the core's
port of QuartiCal's `complex` solver, then compare.

In [ ]:
counts = {k: int(ds.state[k].sum()) for k in ("train", "validation", "test")}
archived = experiment.fixture_pilot()
check = gains.solve(ds)  # 2 integrations x 32 channels per solution, 50 iterations
pd.DataFrame({
    "this notebook": [counts["train"], counts["validation"], counts["test"],
                      check["metrics"]["tv"], check["metrics"]["losses"]["validation"],
                      check["metrics"]["losses"]["test"]],
    "archive (results/gain-tv-flagging.json)": [73720, 9115, 9169, archived["baseline TV (gauge)"],
                                                archived["validation MSE"], archived["test MSE"]],
}, index=["train cells", "validation cells", "test cells", "gain TV", "validation MSE", "test MSE"])

The split counts are equal and the TV and held-out errors agree to about ten
significant figures (the core solver is a bit-for-bit port, research/core-phase2.md).
Only now do we look at the data.

## Amplitudes, flags and weights

A *waterfall* is one baseline's visibilities as an image: time down, frequency
across. Grey cells are flagged. **What to look for:** the smooth bandpass
shape (the receiver response, which the gains will absorb), vertical stripes
(narrow-band radio-frequency interference, RFI, which is constant in time),
the wide flagged block around 1150–1300 MHz (satellite and aircraft
navigation bands that are flagged before the data reach us), and baselines
that look unlike the others.

In [ ]:
fig = flags.waterfall_grid(ds, "amplitude", "XX")
fig

The fraction of flagged cells per channel tells the same story in one line. The
shaded span is the training window (channels 1024–3071).

In [ ]:
flags.flag_fraction_by_channel(ds)

In [ ]:
flags.flag_fraction_by_antenna(ds)

**Weights** are the inverse noise variances the solver and imager use. Real
weights vary with time and frequency; a weight that is wrong by a constant
factor does not change the solution, but one that is wrong *relative to other
cells* does. Here they come from the archive's `WEIGHT_SPECTRUM`.

In [ ]:
flags.waterfall(ds, ds.baselines()[0], "weight", "XX")

## Rank: which gains can the data identify?

The measurement equation for baseline $pq$ is

$$V_{pq} = G_p\, M_{pq}\, G_q^{H} + \text{noise},$$

with a 2×2 complex gain $G_p$ per antenna and solution bin (8 real numbers) and
the model $M_{pq}$. Linearise around a gain point and stack the derivatives into a
Jacobian $J$; the **normal matrix** $J^T W J$ (32×32 for four stations) says
which combinations of gain parameters change the predicted data. Its **rank**
counts the directions the data constrain.

For an unpolarized model ($M \propto I$) the product $G_p M G_q^H$ does not
change under $G_p \to G_p U$ for any unitary $U$, the same for every antenna. That
common U(2) has 4 real parameters, so with $n$ active stations the best possible
rank is $8n-4$. A bin is

- **identified** when its rank equals $8n-4$;
- **underidentified** when it is lower: some gain change is invisible in the data;
- **over-rank** when it is higher, which means the model is not scalar
  (cross-hand model terms break the gauge), not that the data are better;
- **no data** when it has no usable cells.

The native gauge-aware step solver also returns a **status code** for each bin:

In [ ]:
survey.status_table()

We survey every bin of the solution grid (2 integrations × 32 channels), using
all clean cross-correlation cells (the survey's definition; the solve itself
sees only the training part).

In [ ]:
table = survey.identifiability(ds)
summary = survey.rank_summary(table)
pd.Series(summary, name="fixture").to_frame()

In [ ]:
survey.rank_map(table)

**Why do some bins fail?** Look at the baseline graph of each failing bin. If
the baselines with data form a graph *without an odd cycle* (a single baseline,
a path, a star: any tree, or more generally any bipartite graph), split the
stations into the two sides of the graph and apply $G_p \to G_p A$ on one side
and $G_q \to G_q A^{-H}$ on the other, for *any* invertible $A$: every
$G_p G_q^H$ is unchanged. That is 8 free parameters instead of the gauge's 4,
so the rank is $8n-8$. A triangle closes the loop and forces $A$ to be
unitary. The table groups failures by their edge list; the drawings show an
identified bin and the failing shapes.

In [ ]:
survey.failure_shapes(table)

In [ ]:
survey.support_graph(ds, table, survey.example_bins(table))

### What a professional checks before trusting the survey

1. **The gain point.** Rank is computed at identity gains; identity can be
   special, so repeat at a fixed random point. The counts should not move.
2. **The crosscheck call.** The frozen LUNO protocol called the native step as
   `anchored_response(H, 0, H, 0)`, which removes all curvature it keeps and
   returns status 4 on *every* identified bin. It went unnoticed because only
   status 3 was compared; the post-hoc audit (research/flagging-luno-support.md)
   uses `(H, 0, 0, 0)`, which this notebook uses by default. Here is the frozen
   call, so you can see the trap:

In [ ]:
random_point = survey.rank_summary(survey.identifiability(ds, point="random"))
frozen_call = survey.rank_summary(survey.identifiability(ds, crosscheck="frozen"))
pd.DataFrame({"identity (default)": summary, "random gain point": random_point,
              "frozen crosscheck call": frozen_call}).loc[
    ["identified bins", "underidentified bins", "no data bins", "native status counts",
     "status 3 agrees with rank label"]]

## The same survey on the LUNO slice

The LUNO calibrator scans (PKS 1934−638, row chunk 0, stations 0–3, channels
1024–3071) are the project's real-data set. The frozen survey found 432 bins with
data: 389 identified and 43 underidentified, all of them trees. The cell below
reruns it from a local copy of the slice and compares with the archive. It is
skipped when no copy is present (fetching the slice takes about 750 MB:
`data.load_luno(cache_dir=CACHE, fetch=True)`).

In [ ]:
if LUNO:
    luno = data.load_luno()
    luno_table = survey.identifiability(luno)
    mine = survey.rank_summary(luno_table)
    archived = survey.archived_luno_support()
    display(pd.DataFrame({
        "this notebook": [mine["bins with data"], mine["identified bins"],
                          mine["underidentified bins"], mine["no data bins"],
                          mine["identified cells"], mine["underidentified cells"]],
        "archive": [archived["bins_with_train"], archived["fully_identified_bins"],
                    archived["underidentified_bins"], archived["no_train_bins"],
                    archived["fully_identified_cells"], archived["underidentified_cells"]],
    }, index=["bins with data", "identified", "underidentified", "no data",
              "identified cells", "underidentified cells"]))
    display(survey.failure_shapes(luno_table))
    display(survey.rank_map(luno_table))
else:
    print("skipped: no local LUNO slice (set RANG_LUNO_SLICE and RANG_LUNO_EXTRA, "
          "or fetch it with data.load_luno(cache_dir=CACHE, fetch=True))")

## What to take away

- Most bins with data are identified on both datasets; the failures are bins
  where flagging left a tree of baselines. Flagging more can turn an identified
  bin into an underidentified one: rank is the first thing a flagging method
  can break.
- No-data bins come from frequencies flagged on every baseline (the RFI bands),
  not from time.
- Rank says what *can* be solved; it says nothing about how well. The next
  notebooks solve, flag and image.